# Pre-MS stars

This are used to prepare and launch the simulations related to the paper on detectability of p-mode oscillations in pre-main sequence stars.

- Author: Nicholas Jannsen
- Last check: 30-08-2025
- platosim --version: 3.7.0-135-g178ce335 
- git log: b78dfc3775580970209e409a5d83af572a7d9c94 

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# PlatoSim libraries
import platosim.plot      as pt
import platosim.mocka     as mk
import platosim.utilities as ut
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_notebook
setup_notebook()

from IPython.display import display, HTML
display(HTML("<style>.container {width:80% !important; }</style>"))

In [ ]:
hdir = Path(os.getenv('PLATO_PROJECT_HOME')) / 'inputfiles/data_picsim'
path = Path(os.getenv('PLATO_WORKDIR')) / 'pre-ms'
idir = path / 'input'
sdir = path / 'simulations' 

---
## 1. Stellar catalogue
---

In [ ]:
# Load PLATO-CS catalogue
df = pd.read_feather(hdir / 'starcat_PlatoCS_NCAM_LOPS2.ftr')

### Catalogue from June 2025

In [ ]:
# Load Pre-MS catalogue
dt = pd.read_csv(idir / 'targets_june_2025.txt', sep=' ', comment='#',
                 names=['ID', 'gaiaDR3', 'Pmag', 'Gmag', 'ra', 'dec'])
dt.head()

In [ ]:
fig, ax = pt.plotPlatoFOV('LOPS2', system='galactic', fovSize=30,
                          raStars=dt.ra, decStars=dt.dec, c=dt.Pmag,
                          ncamStars=True, showGroups=True, showFcamFOV=False,
                          clabel=r'$\mathcal{P}$ [mag]', s=50, lw=0.2, figsize=(9,9))
# fig.savefig(f'{pdir}/starcat_sky_EBs.png', bbox_inches='tight', dpi=200)

In [ ]:
# Create target catalogue
dt.reset_index(drop=True, inplace=True)
dt.to_feather(idir / 'starcat_PreMS_targets.ftr')

In [ ]:
# Create contaminant catalogue
dc = ut.getContaminants(dt, df, column='gaiaDR3')
dc.to_feather(idir / 'starcat_PreMS_contaminants.ftr')

### Catalogue from November 2025

In [ ]:
# Load Pre-MS catalogue
filename = 'PreMainSequeceLegacy'
dt = pd.read_csv(idir / 'targets_november_2025.csv')
dt.head()

In [ ]:
fig, ax = pt.plotPlatoFOV('LOPS2', system='galactic', fovSize=30,
                          raStars=dt.ra, decStars=dt.dec, c=dt.Pmag, ncamStars=True,
                          clabel=r'$\mathcal{P}$ [mag]', s=50, lw=0.2, figsize=(9,9))
# fig.savefig(pdir / f'starcat_sky_{filename}.png', bbox_inches='tight', dpi=200)

In [ ]:
# Create target catalogue
dt.reset_index(drop=True, inplace=True)
dt.to_feather(idir / f'starcat_{filename}_targets.ftr')

In [ ]:
# Create contaminant catalogue
dc = ut.getContaminants(dt, df, column='gaiaDR3')
dc.to_feather(idir / f'starcat_{filename}_contaminants.ftr')

---
## 2. Post-processing
---

We ran a few test simulations saving the default photometry to the HDF5 file:
```
platonium 1 2 1 1 --project pre-ms --varsource_test.txt 
```

In [ ]:
# Load file to test detrending with
filename = sdir / 'test/hdf5/000000001/000000001_Ncam2.1_Q1.hdf5'

# Load a single mission quarter light curve
lc = LightCurve(filename, mode='single')
dt = lc.star()
dt

In [ ]:
# Show a quarter light curve
lc.plot(flux_unit='ppt', median_filter=1, input_model=False, figsize=(9,4));

### 2.1. Test detrending

In [ ]:
# Generate plot to save
df = lc.detrend(model='poly', replace=True, plot=True)

### 2.2. Test outlier rejection

In [ ]:
# Remove outliers (due to photon noise and cosmic ray hits)
df = lc.clip(model='wotan', sigma_lower=3, sigma_upper=3, flux_unit='ppt', plot=True, replace=True)

### 2.3. Test error estimation

In [ ]:
df = lc.flux_error(flux_unit='ppt', plot=True)

---
## 3. Analysis of simulations
---

In [ ]:
# Initialise light curve object
star = '000000001'
fdir = sdir / 'test/ftr' / star
lcs = LightCurve(fdir, mode="multi")

In [ ]:
# Create table with simulation statistics
df = lcs.stat_sim_table(ofile=fdir / f'lc_{star}.tab', clean=True)
df.head()

In [ ]:
# Show all light curves
fig, ax = lcs.plot_multi(suffix='ftr', group=False, camera=1, quarter=False, 
                         flux_median=144, alpha=0.5, figsize=(9,5))

In [ ]:
# Post-processing into final light curve
lc = lcs.merge(suffix='ftr',
               flux_group_mean=True,
               binsize=50,
               clip_sigma=3,
               flux_stitch=False,
               flux_offset=True,
               flux_error=False,
               ofile= fdir / f'lc_{star}.ftr'
              )

In [ ]:
# Show result
lc.plot(input_model=True, flux_unit='ppt', flux_error=False, median_filter=10, alpha=0.1, figsize=(9,6));

---
## Final data products
---

In [ ]:
# The following file has been reduced on the VSC
filename = 'test/final/lc_000000001.ftr'
lc = LightCurve(sdir / filename, mode="final")

# Introduce gaps from file
inputFileGap = idir / 'instrumentGAP.tab'
lc.gaps(inputFileGap, replace=True, plot=False);

# Show a quarter light curve
lc.plot(flux_unit='ppt', median_filter=1, input_model=True, alpha=0.1, figsize=(9,6));